# BCI Model Performance Comparison

This notebook dynamically aggregates cross-validation fold results from the motor imagery and motor movement pipelines, computes statistical metrics, and generates a comparative visualization.

### Key Visual Components:
* **Mean Accuracy Bars:** Shows the average generalization performance across all cross-validation folds.
* **Error Bars (Standard Deviation):** Represents performance variance and model stability across different data splits.
* **Chance Level (50% Threshold):** The red dashed line indicating random-guess performance for binary classification.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# 1. Automatically fetch the fold results generated by your notebooks
try:
    with open("bci_scores.json", "r") as f:
        scores_data = json.load(f)
    imagery_folds = np.array(scores_data["imagery_folds"])
    movement_folds = np.array(scores_data["movement_folds"])
except FileNotFoundError:
    raise RuntimeError("Run your imagery and movement notebooks first to generate 'bci_scores.json'!")

# 2. Perform the math dynamically (Mean & Standard Deviation)
models = ['Motor Imagery\n(Mental)', 'Motor Movement\n(Physical)']
mean_accuracies = [np.mean(imagery_folds), np.mean(movement_folds)]
std_devs = [np.std(imagery_folds), np.std(movement_folds)]

# 3. Plot a clean bar chart with error bars
fig, ax = plt.subplots(figsize=(6, 4.5))
colors = ['#4C72B0', '#55A868']

bars = ax.bar(
    models, 
    mean_accuracies, 
    yerr=std_devs, 
    capsize=5, 
    color=colors, 
    width=0.5, 
    alpha=0.9, 
    edgecolor='black', 
    linewidth=1
)

# 4. Formatting and Thresholds
ax.axhline(y=50, color='red', linestyle='--', linewidth=1.5, label='Chance Level (50%)')
ax.set_ylabel('Mean Cross-Validation Accuracy (%)', fontsize=11, fontweight='bold')
# Give the Y-axis a bit more head room so the labels don't get clipped at the top
ax.set_ylim(0, 105)
ax.set_title('BCI Model Performance Comparison', fontsize=12, fontweight='bold', pad=12)
ax.legend(loc='upper left', frameon=True)

# 5. Add percentage labels safely *above* the error bar caps (no more overlapping)
for bar, std in zip(bars, std_devs):
    yval = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0, 
        yval + std + 4,  # Dynamically clears the top error cap
        f"{yval:.2f}%", 
        ha='center', 
        va='bottom', 
        fontweight='bold',
        fontsize=11
    )

plt.tight_layout()
plt.show()